In [26]:
import numpy as np
import faiss
import ollama
print("Ollama library loaded successfully")
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

Ollama library loaded successfully


In [2]:
file_path = Path("insurance_claims_records.txt")
text = file_path.read_text(encoding="utf-8")

raw_chunks = text.split("---")
chunks = [chunk.strip() for chunk in raw_chunks if chunk.strip()]
print("Chunks loaded:", len(chunks))

Chunks loaded: 13001


In [3]:
embeddings = np.load("insurance_embedding.npy")
print("Embeddings shape:", embeddings.shape)

Embeddings shape: (13001, 384)


In [4]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("Vectors in index:", index.ntotal)

Vectors in index: 13001


In [5]:
#Load Embedding Model (for queries only)
model = SentenceTransformer("all-MiniLM-L6-v2")

In [6]:
#Query
query = "Customer aged 45 with high insurance claim amount"
query_embedding = model.encode([query]).astype("float32")

In [7]:
#Retreive_records
k= 5

distances, indices = index.search(query_embedding, k)
retrieved_chunks = [chunks[idx] for idx in indices[0]]

In [8]:
#Build Context
context = "\n\n".join(retrieved_chunks)

In [9]:
#Build Prompt
prompt = f"""
You are an insurance analysis assistant.

Context:
{context}

Question:
{query}

Answer:
"""

In [10]:
#Send to LLM
response=ollama.chat(
    model="phi3:mini",
    messages=[{"role":"user","content":prompt}]
)

print(response["message"]["content"])


The customer who fits the criteria of being 45 years old with a high insurance claim amount is described in insurance record 145. This profile shows a male customer, aged 45, who works as a Doctor, holds a PhD, is married, and has an annual income of 120715.0. Notably, the insurance claim amount for this customer stands out at 80927.0, which is considerably high compared to the other records mentioned.


In [28]:
def ask_insurance_rag(question):
    query_embedding = model.encode([question]).astype("float32")
    distances, indices = index.search(query_embedding, 5)
    retrieved_chunks = [chunks[idx] for idx in indices[0]]
    retrieved_embeddings = embeddings[indices[0]]
#Re-rank using cosine similarity
    similarity_scores = cosine_similarity(query_embedding, retrieved_embeddings)[0]
    ranked_pairs = sorted(
        zip(similarity_scores, retrieved_chunks),
        reverse=True
    )
#Select top 3 after re-ranking
    top_chunks = [chunk for _, chunk in ranked_pairs[:3]]
    
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are an insurance analysis assistant.

Context:
{context}

Question:
{question}

Answer:
"""

    response = ollama.chat(
        model="phi3:mini",
        messages=[{"role":"user","content":prompt}]
    )
    return response["message"]["content"]

In [30]:
ask_insurance_rag("Customer aged 45 with high claim amount")

'The customer that matches the given criteria is from insurance record 5951. This customer is a 45-year-alive Male CEO with a PhD education level and single marital status. He earns an annual income of 183277.0 and has a recorded insurance claim amount of 3424.0, which is considered high.'